# Supplementary Figure 8

Panel a compares accepted peptides of 9-50 residues in four PREDICT database arms on the same 296 acquisitions.
Panels b-d show what changes in a fixed FastaLake reporting dictionary when peptide evidence that the matched
metagenome search did not accept is removed (296 PREDICT and 295 CapScan acquisitions). The v30 figure is drawn on
one canvas in house style; the producer drawing is run unchanged alongside it and every plotted array is compared.

Run from the repository root: `python paper.py build SFig08`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
from IPython.display import Image, display

## Panel a

Accepted peptides (9-50 residues, peptide q ≤ 0.01, I and L merged) per acquisition in four PREDICT arms: full
FastaLake including cohort metagenomes (`FL_cohortMG`), specimen-matched metagenome (`MG`), public-sequence FastaLake
(`FL_public`) and curated specimen MG (`MG_own_curated`). Supports v30 Results (v30f line 67, v28d line 123): "these
curated specimen-metagenome databases identified a median 1.5% fewer peptides than the unselected metagenome search
(fewer in 177 of 296 acquisitions), whereas public-sequence FastaLake exceeded them by a median 10.6%, with higher
peptide counts in 292 of 296 acquisitions". n = 296 acquisitions per arm. No test: medians of per-acquisition gains
and counts of acquisitions.

In [ ]:
res = panels.render_v30(SD, OUT)
counts, abl = res["counts"], res["abl"]
panels.crop_px(res["png"], res["axes_bbox_px"]["a"], OUT / "panel_a.png")
display(Image(url="../../build/SFig08/panel_a.png", embed=False))   # linked, not embedded (keeps the notebook small)
gain_curated_vs_mg = panels.stat_a_paired_gain(counts, "MG_own_curated", "MG")
gain_public_vs_curated = panels.stat_a_paired_gain(counts, "FL_public", "MG_own_curated")
arms = panels.stat_a_n_arms(counts)
for k, v in [("curated MG vs MG", gain_curated_vs_mg), ("public FL vs curated MG", gain_public_vs_curated), ("arms", arms)]:
    print(k, v)

## Panel b

LFQ intensity removed (%) when features whose peptide was not accepted by the same-acquisition metagenome search are
dropped. Fixed reporting dictionary. Supports the legend (v30f line 236, v28d line 441); the manuscript prints no value
from this panel. n = 296 PREDICT and 295 CapScan acquisitions. No test: bars are medians.

In [ ]:
panels.crop_px(res["png"], res["axes_bbox_px"]["b"], OUT / "panel_b.png")
display(Image(url="../../build/SFig08/panel_b.png", embed=False))
bcd = panels.stat_bcd_medians(abl)
print(bcd[['n', 'intensity_removed_percent']].to_string())

## Panel c

KEGG orthologs (KOs) losing all support (%) after the same removal. Supports the legend (v30f line 236); no value
printed. n = 296 PREDICT and 295 CapScan acquisitions. No test: bars are medians.

In [ ]:
panels.crop_px(res["png"], res["axes_bbox_px"]["c"], OUT / "panel_c.png")
display(Image(url="../../build/SFig08/panel_c.png", embed=False))
print(bcd[['n', 'KO_support_lost_percent']].to_string())

## Panel d

Bray-Curtis distance between the normalized all-evidence and shared-only KO profiles. Supports the legend (v30f line
236); no value printed. n = 296 PREDICT and 295 CapScan acquisitions. No test: bars are medians.

In [ ]:
panels.crop_px(res["png"], res["axes_bbox_px"]["d"], OUT / "panel_d.png")
display(Image(url="../../build/SFig08/panel_d.png", embed=False))
print(bcd[['n', 'BrayCurtis']].to_string())

## Checks and numbers

Pixel checks of the v30 composite (PNG at 300 dpi, PDF rasterised at 300 dpi) against the published v30 render, the
restyle-versus-producer value checks, and every number in `NUMBERS.tsv` in file order.

In [ ]:
from tools.check_pixels import compare
if not fl._HAVE_STYLE:
    print("pixel checks SKIPPED: fonts absent")
else:
    for name, new, exp, kw in [("SFig08_v35", res["png"], "SFig08_v35.png", {}),
                               ("SFig08_v35_pdf", res["pdf"], "SFig08_v35.pdf", {"dpi": 300})]:
        r = compare(new, UNIT / "expected" / exp, UNIT / "checks" / name, **kw)
        print(name, r["verdict"], r.get("differing_pixels"), r.get("max_channel_difference"))
print(sum(c["passed"] for c in res["checks"]), "of", len(res["checks"]), "value checks passed")
rows = [
    ("123", -gain_curated_vs_mg["median_gain_percent"]),
    ("123", gain_curated_vs_mg["n_lower"]),
    ("123", gain_curated_vs_mg["n"]),
    ("123", gain_public_vs_curated["median_gain_percent"]),
    ("123", gain_public_vs_curated["n_higher"]),
    ("123", gain_public_vs_curated["n"]),
    ("441", arms["n_arms_word"]),
    ("441", arms["n_acquisitions"]),
    ("441", int(bcd.loc["PREDICT", "n"])),
    ("441", int(bcd.loc["CapScan", "n"])),
]
for line, value in rows:
    print(line, repr(value))